In [1]:
from torchgeo.datasets import RasterDataset

import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import numpy as np
import os

In [6]:
def make_cmd(path, name):
    return ["C:\\Program Files\\R\\R-4.4.1\\bin\\Rscript.exe",'pseudo_absence.R', 
           path, name]

files_list = ["inputs\\nst_pest_sightings\\bean_leaf_beetle_0003221-260623161305970\\0003221-260623161305970.csv", 
              "inputs\\nst_pest_sightings\\bird_cherry_aphid_0003207-260623161305970\\0003207-260623161305970.csv",
              "inputs\\nst_pest_sightings\\black_cutworm_0003212-260623161305970\\0003212-260623161305970.csv",
            #   "nst_pest_sightings\\corn_leaf_aphid_0003215-260623161305970\\0003215-260623161305970.csv",
              "inputs\\nst_pest_sightings\\differential_grasshopper0032387-260623161305970\\0032387-260623161305970.csv",
            #   "nst_pest_sightings\\european_corn_borer0003385-260623161305970\\0003385-260623161305970.csv",
              "inputs\\nst_pest_sightings\\green_stink_0005409-260623161305970\\0005409-260623161305970.csv",
            #   "nst_pest_sightings\\hessian_fly_0003394-260623161305970\\0003394-260623161305970.csv",
              "inputs\\nst_pest_sightings\\japanese_beetle_0003383-260623161305970\\0003383-260623161305970.csv",
              "inputs\\nst_pest_sightings\\northern_corn_rootworm_0003360-260623161305970\\0003360-260623161305970.csv",
            #   "nst_pest_sightings\\reg_legged_grasshopperobservations-752840.csv\\observations-752840.csv",
              "inputs\\nst_pest_sightings\\seedcorn_maggot_0003200-260623161305970\\0003200-260623161305970.csv",
              "inputs\\nst_pest_sightings\\southern_green_stink_0003348-260623161305970\\0003348-260623161305970.csv",
              "inputs\\nst_pest_sightings\\three_cornered_alfalfa_0003390-260623161305970\\0003390-260623161305970.csv",
              "inputs\\nst_pest_sightings\\true_armyworm0032387-260623161305970\\0032387-260623161305970.csv",
              "input\\nst_pest_sightings\\two_striped_grasshopper0032397-260623161305970\\0032397-260623161305970.csv",
              "inputs\\nst_pest_sightings\\western_corn_rootworm_0003372-260623161305970\\0003372-260623161305970.csv"
              ]

ras_feats = ["inputs/chelsa_clim/current/CHELSA_bio02_1981-2010_V.2.1.tif", 
             "inputs/chelsa_clim/current/CHELSA_bio04_1981-2010_V.2.1.tif", 
             "inputs/chelsa_clim/current/CHELSA_bio06_1981-2010_V.2.1.tif",
             "inputs/chelsa_clim/current/CHELSA_bio14_1981-2010_V.2.1.tif",
             "inputs/chelsa_clim/current/CHELSA_bio15_1981-2010_V.2.1.tif",
             "inputs/chelsa_clim/current/CHELSA_bio19_1981-2010_V.2.1.tif",
             "inputs/chelsa_clim/current/CHELSA_fgd_1981-2010_V.2.1.tif",
             "inputs/chelsa_clim/current/CHELSA_scd_1981-2010_V.2.1.tif"]

training_feature_names = ["bio02", "bio04", "bio06", "bio14", "bio15", "bio19", "fgd", "scd"]

command_blb = make_cmd(files_list[0], "beanleafbeetle")
command_bca = make_cmd(files_list[1], "bircherryaphid")
command_bc = make_cmd(files_list[2], "blackcutworm")
command_dg = make_cmd(files_list[3], "differentialgrasshopper") 
command_gs = make_cmd(files_list[4], "greenStink")
command_jb = make_cmd(files_list[5], "japanesebeetle")
command_ncr = make_cmd(files_list[6], "northerncornrootworm")
command_sm = make_cmd(files_list[7], "seedcornmaggot")
command_sgs = make_cmd(files_list[8], "southerngreenstink")
command_tca = make_cmd(files_list[9], "threecornerneredalfalfa")
command_ta = make_cmd(files_list[10], "truearmyworm")
command_tsg = make_cmd(files_list[11], "two-striped-grasshopper")
command_wcr = make_cmd(files_list[12], "westcornrootworm")

command_list = [command_blb, command_bca, command_bc, command_dg, command_gs, command_jb, command_ncr, command_sm, command_sgs, command_tca, command_ta, command_tsg, command_wcr]

min_lon = -170
max_lon = -52
min_lat = 24
max_lat = 83.5

In [3]:
ncr_gdf_mid = gpd.read_file('outputs/data/' + command_blb[3] + '/mid.shp')

In [8]:
ncr_gdf_mid.sample(5)

,CLASS,geometry
1292,1.0,POLYGON EMPTY
707,0.0,POLYGON EMPTY
1142,0.0,POLYGON EMPTY
322,0.0,POLYGON EMPTY
1301,0.0,POLYGON EMPTY


In [9]:
from pathlib import Path

import geopandas as gpd
import rasterio
import numpy as np


raster_dir = Path("inputs/chelsa_clim/current")
shp_path = Path("outputs/data/beanleafbeetle/mid.shp")
mask_dir = Path("outputs/masks")
mask_dir.mkdir(parents=True, exist_ok=True)

gdf_original = gpd.read_file(shp_path)

# If your shapefile has a class column, put its name here.
# If you want binary masks, set class_column = None.
class_column = None
# class_column = "class_id"


for raster_path in raster_dir.glob("*.tif"):
    print(f"Processing {raster_path.name}")

    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        raster_transform = src.transform
        raster_width = src.width
        raster_height = src.height
        raster_bounds = src.bounds
        raster_meta = src.meta.copy()

        # Reproject points to match raster CRS
        gdf = gdf_original.to_crs(raster_crs)

        # Keep only points inside raster bounds
        gdf = gdf.cx[
            raster_bounds.left:raster_bounds.right,
            raster_bounds.bottom:raster_bounds.top
        ]

        # Create empty mask
        dtype = "uint8" if class_column is None else "uint16"
        mask = np.zeros((raster_height, raster_width), dtype=dtype)

        for _, row in gdf.iterrows():
            geom = row.geometry

            if geom is None or geom.is_empty:
                continue

            x = geom.x
            y = geom.y

            # Convert map coordinates to raster row, col
            row_idx, col_idx = src.index(x, y)

            # Safety check
            if 0 <= row_idx < raster_height and 0 <= col_idx < raster_width:
                if class_column is None:
                    value = 1
                else:
                    value = int(row[class_column])

                mask[row_idx, col_idx] = value

    mask_meta = raster_meta.copy()
    mask_meta.update({
        "count": 1,
        "dtype": dtype,
        "nodata": 0,
        "compress": "lzw"
    })

    mask_path = mask_dir / f"{raster_path.stem}_pointmask.tif"

    with rasterio.open(mask_path, "w", **mask_meta) as dst:
        dst.write(mask, 1)

    print(f"  Saved {mask_path}")

Processing CHELSA_bio02_1981-2010_V.2.1.tif
  Saved outputs\masks\CHELSA_bio02_1981-2010_V.2.1_pointmask.tif
Processing CHELSA_bio04_1981-2010_V.2.1.tif
  Saved outputs\masks\CHELSA_bio04_1981-2010_V.2.1_pointmask.tif
Processing CHELSA_bio05_1981-2010_V.2.1.tif
  Saved outputs\masks\CHELSA_bio05_1981-2010_V.2.1_pointmask.tif
Processing CHELSA_bio06_1981-2010_V.2.1.tif
  Saved outputs\masks\CHELSA_bio06_1981-2010_V.2.1_pointmask.tif
Processing CHELSA_bio13_1981-2010_V.2.1.tif
  Saved outputs\masks\CHELSA_bio13_1981-2010_V.2.1_pointmask.tif
Processing CHELSA_bio14_1981-2010_V.2.1.tif
  Saved outputs\masks\CHELSA_bio14_1981-2010_V.2.1_pointmask.tif
Processing CHELSA_bio15_1981-2010_V.2.1.tif
  Saved outputs\masks\CHELSA_bio15_1981-2010_V.2.1_pointmask.tif
Processing CHELSA_bio19_1981-2010_V.2.1.tif
  Saved outputs\masks\CHELSA_bio19_1981-2010_V.2.1_pointmask.tif
Processing CHELSA_fgd_1981-2010_V.2.1.tif
  Saved outputs\masks\CHELSA_fgd_1981-2010_V.2.1_pointmask.tif
Processing CHELSA_scd_1

In [7]:
# Optional: fix invalid geometries
ncr_gdf_mid["geometry"] = ncr_gdf_mid.geometry.buffer(0)

# Column in shapefile containing class labels
# If you just want binary labels, set class_column = None
class_column = "class_id"   # change this to your column name
# class_column = None


for raster_path in ras_feats:
    print("Processing")

    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        raster_transform = src.transform
        raster_width = src.width
        raster_height = src.height
        raster_bounds = src.bounds
        raster_meta = src.meta.copy()

    # Reproject shapefile to match current raster CRS
    gdf = ncr_gdf_mid.to_crs(raster_crs)

    # Optional but recommended: keep only features intersecting raster bounds
    gdf = gdf.cx[
        raster_bounds.left:raster_bounds.right,
        raster_bounds.bottom:raster_bounds.top
    ]

    if len(gdf) == 0:
        print(f"  No shapefile features overlap")

        # Create empty mask
        shapes = []
    else:
        if class_column is None:
            # Binary mask: burn all geometries as 1
            shapes = [(geom, 1) for geom in gdf.geometry if geom is not None]
            dtype = "uint8"
        else:
            # Multi-class mask: burn values from class_column
            shapes = [
                (geom, int(value))
                for geom, value in zip(gdf.geometry, gdf[class_column])
                if geom is not None
            ]
            dtype = "uint16"

    mask = rasterize(
        shapes=shapes,
        out_shape=(raster_height, raster_width),
        transform=raster_transform,
        fill=0,
        dtype=dtype
    )

    mask_meta = raster_meta.copy()
    mask_meta.update({
        "count": 1,
        "dtype": dtype,
        "nodata": 0,
        "compress": "lzw"
    })

    mask_path = "outputs/cnn"
    full_mask_path = mask_path + "/labels_mask_.tif"

    mask_path = full_mask_path / "h_mask.tif"

    with rasterio.open(mask_path, "w", **mask_meta) as dst:
        dst.write(mask, 1)

    print(f"  Saved {mask_path}")

Processing
  No shapefile features overlap


NameError: name 'dtype' is not defined

In [ ]:
mask = rasterize(
    shapes=shapes,
    out_shape=(raster_height, raster_width),
    transform=raster_transform,
    fill=0,
    dtype="uint8"
)

mask_meta = raster_meta.copy()
mask_meta.update({
    "count": 1,
    "dtype": "uint8",
    "nodata": 0
})


os.makedirs(mask_path, exist_ok=True)
with rasterio.open(full_mask_path, "w", **mask_meta) as dst:
    dst.write(mask, 1)

In [ ]:
from torchgeo.datasets import RasterDataset
from torchgeo.samplers import RandomGeoSampler
from torchgeo.datasets.utils import stack_samples
from torch.utils.data import DataLoader


class ImageDataset(RasterDataset):
    filename_glob = "image.tif"
    is_image = True


class MaskDataset(RasterDataset):
    filename_glob = "labels_mask.tif"
    is_image = False


image_ds = ImageDataset("path/to/image_folder")
mask_ds = MaskDataset("path/to/mask_folder")

dataset = image_ds & mask_ds